In [1]:
import jieba
import pandas as pd
from sklearn import svm #在问号？处填入需要用到的分类器
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import classification_report, accuracy_score


In [ ]:
train_data = pd.read_csv('data/chnsenticorp/train.tsv', sep='\t')
test_data = pd.read_csv('data/chnsenticorp/test.tsv', sep='\t')
label_map = {0: '负面', 1: '正面'}
x_train, y_train = train_data.text_a.values, train_data.label.values
x_test, y_test = test_data.text_a.values, test_data.label.values


In [ ]:
class SVM_Classifier(object):
    
    def __init__(self, use_chi=False):
        
        self.use_chi = use_chi # 是否使用卡方检验做特征选择
        # 支持向量机分类器
        self.model = svm.SVC(C=1.0, kernel='linear', degree=3, gamma='auto')
        # 使用tf-idf 特征提取
        self.feature_processor = TfidfVectorizer(tokenizer=jieba.cut)
        # 卡方检验特征选择
        if use_chi:
            self.feature_selector = SelectKBest(chi2, k=10000) # 34814 -> 10000
        
    def fit(self, x_train, y_train, x_test, y_test):
        
        x_train_fea = self.feature_processor.fit_transform(x_train)
        if self.use_chi:
            x_train_fea = self.feature_selector.fit_transform(x_train_fea, y_train)
        self.model.fit(x_train_fea, y_train)
        
        train_accuracy = self.model.score(x_train_fea, y_train)
        print("训练集准确率：{}".format(round(train_accuracy, 3)))
        
        x_test_fea = self.feature_processor.transform(x_test) #从测试集中提取特征
        if self.use_chi:
            x_test_fea = self.feature_selector.transform(x_test_fea)
        y_predict = self.model.predict(x_test_fea)

        test_accuracy = accuracy_score(y_test, y_predict)
        print("测试集准确率：{}".format(round(test_accuracy, 3)))
        print('测试集评估矩阵：')
        print(classification_report(y_test, y_predict, target_names=['负面', '正面']))
        
    def single_predict(self, text):
        text_preprocess = [" ".join(jieba.cut(text))]
        text_fea = self.feature_processor.transform(text_preprocess)
        if self.use_chi:
            text_fea = self.feature_selector.transform(text_fea)
        predict_idx = self.model.predict(text_fea)[0]
        predict_label = label_map[predict_idx]
        
        return predict_label


In [ ]:
svm_classifier = SVM_Classifier()
svm_classifier.fit(x_train, y_train, x_test, y_test)


In [ ]:
svm_classifier = SVM_Classifier(use_chi=True)
svm_classifier.fit(x_train, y_train, x_test, y_test) #训练分类器


In [ ]:
def feature_analysis():
    feature_names = svm_classifier.feature_processor.get_feature_names()
    feature_scores = svm_classifier.feature_selector.scores_
    fea_score_tups = list(zip(feature_names, feature_scores))
    fea_score_tups.sort(key=lambda tup: tup[1], reverse=True)
    return fea_score_tups

# 查看卡方值top500词
feature_analysis()[:500]


In [ ]:
svm_classifier.single_predict("外观很漂亮，出人意料地漂亮，做工非常好")

In [ ]:
svm_classifier.single_predict("书的内容没什么好说的，主要是纸张、印刷太差，所用的纸非常粗糙比一般的盗版书还要差，裁的也不好。")